In [1]:
import pandas as pd
import numpy as np
from groq import Groq
from sklearn.metrics import classification_report, f1_score
import time
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

# Load all 10 Groq keys
groq_keys = []
for i in range(1, 11):
    key = os.getenv(f"GROQ_KEY_{i}")
    if key:
        groq_keys.append(key)

print(f" Loaded {len(groq_keys)} Groq API keys")

# Key rotation
current_key_index = 0

def get_client():
    return Groq(api_key=groq_keys[current_key_index])

 Loaded 10 Groq API keys


In [2]:
# Load test data only (200 samples for zero-shot)
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')

# Combine headline + content
test_df['text'] = test_df['headline'].fillna('') + ' ' + test_df['content'].fillna('')
test_df = test_df.dropna(subset=['text', 'label'])

# Sample 200 rows (balanced)
authentic_sample = test_df[test_df['label'] == 'authentic'].sample(n=67, random_state=42)
fake_sample = test_df[test_df['label'] == 'fake'].sample(n=67, random_state=42)
ai_fake_sample = test_df[test_df['label'] == 'ai_fake'].sample(n=66, random_state=42)

sample_df = pd.concat([authentic_sample, fake_sample, ai_fake_sample])
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f" Test samples: {len(sample_df)}")
print(sample_df['label'].value_counts())

 Test samples: 200
label
fake         67
authentic    67
ai_fake      66
Name: count, dtype: int64


In [3]:
# Zero-shot classification function
def classify_news(text, retries=3):
    global current_key_index
    
    prompt = f"""তুমি একজন বাংলা সংবাদ বিশেষজ্ঞ।
নিচের বাংলা সংবাদটি পড়ো এবং এটি কোন ধরনের তা বলো।

সংবাদ:
{text[:500]}

শুধু এই তিনটির একটি উত্তর দাও:
- authentic (যদি এটি সত্যিকারের খবর হয়)
- fake (যদি এটি মানুষের লেখা ভুয়া খবর হয়)
- ai_fake (যদি এটি AI দিয়ে তৈরি ভুয়া খবর হয়)

উত্তর (শুধু একটি শব্দ):"""

    for attempt in range(retries):
        try:
            client = get_client()
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0.0
            )
            result = response.choices[0].message.content.strip().lower()
            
            # Extract label
            if 'authentic' in result:
                return 'authentic'
            elif 'ai_fake' in result or 'ai fake' in result:
                return 'ai_fake'
            elif 'fake' in result:
                return 'fake'
            else:
                return 'authentic'  # default
                
        except Exception as e:
            print(f"\nKey {current_key_index+1} failed: {e}")
            current_key_index = (current_key_index + 1) % len(groq_keys)
            time.sleep(2)
    
    return 'authentic'  # default after all retries

print("Zero-shot function ready!")

Zero-shot function ready!


In [4]:
from tqdm import tqdm

# Run zero-shot classification
print("Starting Zero-shot classification...")
predictions = []

for i, row in tqdm(
    sample_df.iterrows(),
    total=len(sample_df),
    desc="Classifying"
):
    pred = classify_news(row['text'])
    predictions.append(pred)
    time.sleep(1)

# Results
print("\n=== Zero-shot Llama 70B Results ===")
print(classification_report(
    sample_df['label'],
    predictions,
    target_names=['authentic', 'fake', 'ai_fake']
))

zs_f1 = f1_score(sample_df['label'], predictions, average='macro')
print(f"Test Macro F1: {zs_f1*100:.2f}%")

Starting Zero-shot classification...


Classifying:  38%|███▊      | 77/200 [04:52<10:00,  4.88s/it]


Key 1 failed: upstream connect error or disconnect/reset before headers. reset reason: remote connection failure, transport failure reason: delayed connect error: Connection refused


Classifying:  94%|█████████▍| 188/200 [12:18<00:55,  4.61s/it]


Key 2 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kjj7ktz8f5nbd9ksbja13n5m` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99596, Requested 967. Please try again in 8m6.431999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Classifying: 100%|██████████| 200/200 [12:38<00:00,  3.79s/it]


=== Zero-shot Llama 70B Results ===
              precision    recall  f1-score   support

   authentic       0.59      0.29      0.39        66
        fake       0.49      0.90      0.63        67
     ai_fake       0.52      0.36      0.42        67

    accuracy                           0.52       200
   macro avg       0.54      0.51      0.48       200
weighted avg       0.54      0.52      0.48       200

Test Macro F1: 48.25%


In [5]:
# Save zero-shot results
import pandas as pd

results_df = pd.DataFrame({
    'text': sample_df['text'].values,
    'true_label': sample_df['label'].values,
    'predicted_label': predictions
})

results_df.to_csv(
    'C:/Users/Riyad/projects/fake_news/zeroshot_results.csv',
    index=False
)

print(" Zero-shot results saved!")
print(f"Macro F1: 48.25%")

 Zero-shot results saved!
Macro F1: 48.25%
